In [2]:
import pandas as pd
import os
import plotly.express as px

In [3]:
#csv file downloaded from https://www.macrotrends.net/1324/s-p-500-earnings-history and added an additional column header eps
df = pd.read_csv(os.path.join("data", "s-p-500-earnings-history.csv"),skiprows=15)
df.tail()

,date,value,eps
1174,2025-10-01,6840.20,234.06
1175,2025-11-01,6849.09,234.06
1176,2025-12-01,6845.50,234.06
1177,2026-01-01,6939.03,234.06
1178,2026-02-01,6878.88,234.06


In [4]:
df.columns = ['date', 'price', 'eps']
df['date'] = pd.to_datetime(df['date'])

In [5]:
df['earningyield'] = df['eps'] / df['price']*100
df.tail()

,date,price,eps,earningyield
1174,2025-10-01,6840.20,234.06,3.421830
1175,2025-11-01,6849.09,234.06,3.417388
1176,2025-12-01,6845.50,234.06,3.419180
1177,2026-01-01,6939.03,234.06,3.373094
1178,2026-02-01,6878.88,234.06,3.402589


In [6]:
df_last_25 = df[df['date'] >= '2005-01-01']

# Create bar chart
fig = px.bar(df_last_25, x='date', y='earningyield', title='S&P 500 Earnings Yield since 2005')
fig.update_layout(yaxis_title='Earnings Yield', xaxis_title='Year', title_y=0.9, margin=dict(t=70))
fig.show()

In [7]:
# Extract year
df['year'] = df['date'].dt.year

# Aggregate per year: average earnings yield and calculate annual return
df_yearly = df.groupby('year').agg(
    avg_earningyield=('earningyield', 'mean'),
    start_price=('price', 'first'),
    end_price=('price', 'last')
).reset_index()

df_yearly['annual_return'] = (df_yearly['end_price'] / df_yearly['start_price']) - 1

df_yearly.tail()

,year,avg_earningyield,start_price,end_price,annual_return
95,2022,4.611074,4515.55,3839.50,-0.149716
96,2023,4.245804,4076.60,4769.83,0.170051
97,2024,3.660655,4845.65,5881.63,0.213796
98,2025,3.625595,6040.53,6845.50,0.133261
99,2026,3.387841,6939.03,6878.88,-0.008668


In [8]:
# Convert year to datetime and set as index
df_yearly['year'] = pd.to_datetime(df_yearly['year'], format='%Y')
df_yearly = df_yearly.set_index('year')

df_yearly.tail()

,avg_earningyield,start_price,end_price,annual_return
year,,,,
2022-01-01,4.611074,4515.55,3839.50,-0.149716
2023-01-01,4.245804,4076.60,4769.83,0.170051
2024-01-01,3.660655,4845.65,5881.63,0.213796
2025-01-01,3.625595,6040.53,6845.50,0.133261
2026-01-01,3.387841,6939.03,6878.88,-0.008668


In [9]:
import plotly.graph_objects as go

# Create figure with bar for earnings yield and line for annual return
fig = go.Figure()

# Bar for earnings yield, color red if annual return is negative
colors = 'blue'
fig.add_trace(go.Bar(
    x=df_yearly.index,
    y=df_yearly['avg_earningyield'],
    name='Average Earnings Yield',
    marker_color=colors
))

# Line for annual return (secondary y-axis)
fig.add_trace(go.Scatter(
    x=df_yearly.index,
    y=df_yearly['annual_return'],
    mode='lines+markers',
    name='Annual Return',
    line=dict(color='green'),
    yaxis='y2'
))

# Update layout
fig.update_layout(
    title='S&P 500 Average Earnings Yield and Annual Return per Year',
    xaxis_title='Year',
    yaxis_title='Earnings Yield (%)',
    yaxis2=dict(
        title='Annual Return',
        overlaying='y',
        side='right'
    ),
    barmode='group'
)

fig.show()

In [10]:
dfyield = pd.read_excel(os.getcwd()+"/data/histretsp.xlsx",
    sheet_name='Returns by year',
    header=None,
    nrows=93,
    skiprows=range(0,18),
    usecols=[0,1,2,3,4,5])

dfyield.columns =["year","SP500ret","3MTBill","10YUSTBond","BAACorpBond","RealEstate"]
dfyield.set_index("year",inplace=True)
dfyield.head()

c:\ProgramData\Anaconda3\envs\trading\lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning:

Cannot parse header or footer so it will be ignored



,SP500ret,3MTBill,10YUSTBond,BAACorpBond,RealEstate
year,,,,,
1928,0.438112,0.0308,0.008355,0.032196,0.014911
1929,-0.082979,0.0316,0.042038,0.030179,-0.020568
1930,-0.251236,0.0455,0.045409,0.005398,-0.043000
1931,-0.438375,0.0231,-0.025589,-0.156808,-0.081505
1932,-0.086424,0.0107,0.087903,0.235896,-0.104664


In [18]:
# Convert dfyield index to datetime for merging
dfyield.index = pd.to_datetime(dfyield.index, format='%Y')

# Merge df_yearly and dfyield on the year index
df_merged = df_yearly.merge(dfyield, left_index=True, right_index=True, how='inner')
df_merged['10YUSTBond'] = df_merged['10YUSTBond'] * 100

# Calculate the difference between earnings yield and 10Y US T-Bond
# This is often referred to as the "yield spread" and can indicate whether stocks are relatively cheap or expensive compared to bonds.
# A positive spread suggests stocks may be undervalued (higher earnings yield than bond yield), while a negative spread suggests stocks may be overvalued.
df_merged['yield_bond_spread'] = df_merged['avg_earningyield'] - df_merged['10YUSTBond']

df_merged.tail()

,avg_earningyield,start_price,end_price,annual_return,smoothed_yield,7yr_return,SP500ret,3MTBill,10YUSTBond,BAACorpBond,RealEstate,yield_bond_spread
year,,,,,,,,,,,,
2016-01-01,4.241817,1940.24,2238.83,0.153893,4.348064,0.714958,0.117731,0.003161,0.690550,0.103651,0.053097,3.551266
2017-01-01,4.275067,2278.87,2673.61,0.173217,4.364782,0.784041,0.216055,0.009341,2.801716,0.097239,0.062131,1.473350
2018-01-01,4.577463,2823.81,2506.85,-0.112246,4.491472,1.346223,-0.042269,0.019363,-0.016692,-0.027626,0.045327,4.594155
2019-01-01,4.621887,2704.10,3230.78,0.194771,4.149165,1.118838,0.312117,0.020625,9.635631,0.153295,0.036916,-5.013744
2020-01-01,3.248145,3225.52,3756.07,0.164485,3.901934,0.831404,0.180232,0.003547,11.331898,0.104115,0.103461,-8.083753


In [11]:
# Smooth the earnings yield with a 3-year rolling average
df_yearly['smoothed_yield'] = df_yearly['avg_earningyield'].rolling(window=3, center=True).mean()

# Calculate 7-year rolling return
df_yearly['7yr_return'] = (df_yearly['end_price'].shift(-6) / df_yearly['end_price']) - 1

df_yearly.tail(10)

,avg_earningyield,start_price,end_price,annual_return,smoothed_yield,7yr_return
year,,,,,,
2017-01-01,4.275067,2278.87,2673.61,0.173217,4.364782,0.784041
2018-01-01,4.577463,2823.81,2506.85,-0.112246,4.491472,1.346223
2019-01-01,4.621887,2704.10,3230.78,0.194771,4.149165,1.118838
2020-01-01,3.248145,3225.52,3756.07,0.164485,3.901934,0.831404
2021-01-01,3.835772,3714.24,4766.18,0.283218,3.898330,NaN
2022-01-01,4.611074,4515.55,3839.50,-0.149716,4.230883,NaN
2023-01-01,4.245804,4076.60,4769.83,0.170051,4.172511,NaN
2024-01-01,3.660655,4845.65,5881.63,0.213796,3.844018,NaN
2025-01-01,3.625595,6040.53,6845.50,0.133261,3.558031,NaN


In [12]:
# Create figure with bar for smoothed earnings yield and line for 7-year return
fig = go.Figure()

# Bar for smoothed earnings yield, color red if 7-year return is negative
colors = 'blue'
fig.add_trace(go.Bar(
    x=df_yearly.index,
    y=df_yearly['smoothed_yield'],
    name='Smoothed Earnings Yield (3-year avg)',
    marker_color=colors
))

# Line for 7-year return (secondary y-axis)
fig.add_trace(go.Scatter(
    x=df_yearly.index,
    y=df_yearly['7yr_return'],
    mode='lines+markers',
    name='7-Year Return',
    line=dict(color='orange'),
    yaxis='y2'
))

# Update layout
fig.update_layout(
    title='S&P 500 Smoothed Earnings Yield and 7-Year Return for Long-Term Performance Perspective',
    xaxis_title='Year',
    yaxis_title='Smoothed Earnings Yield (%)',
    yaxis2=dict(
        title='7-Year Return',
        overlaying='y',
        side='right'
    ),
    barmode='group'
)

fig.show()

In [13]:
dfyield = pd.read_excel(os.getcwd()+"/data/histretsp.xlsx",
    sheet_name='Returns by year',
    header=None,
    nrows=93,
    skiprows=range(0,18),
    usecols=[0,1,2,3,4,5])

dfyield.columns =["year","SP500ret","3MTBill","10YUSTBond","BAACorpBond","RealEstate"]
dfyield.set_index("year",inplace=True)
dfyield.head()

c:\ProgramData\Anaconda3\envs\trading\lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning:

Cannot parse header or footer so it will be ignored



,SP500ret,3MTBill,10YUSTBond,BAACorpBond,RealEstate
year,,,,,
1928,0.438112,0.0308,0.008355,0.032196,0.014911
1929,-0.082979,0.0316,0.042038,0.030179,-0.020568
1930,-0.251236,0.0455,0.045409,0.005398,-0.043000
1931,-0.438375,0.0231,-0.025589,-0.156808,-0.081505
1932,-0.086424,0.0107,0.087903,0.235896,-0.104664


In [14]:
# Convert dfyield index to datetime for merging
dfyield.index = pd.to_datetime(dfyield.index, format='%Y')

# Merge df_yearly and dfyield on the year index
df_merged = df_yearly.merge(dfyield, left_index=True, right_index=True, how='inner')
df_merged['10YUSTBond'] = df_merged['10YUSTBond'] * 100

# Calculate the difference between earnings yield and 10Y US T-Bond
df_merged['yield_bond_spread'] = df_merged['avg_earningyield'] - df_merged['10YUSTBond']

df_merged.tail()

,avg_earningyield,start_price,end_price,annual_return,smoothed_yield,7yr_return,SP500ret,3MTBill,10YUSTBond,BAACorpBond,RealEstate,yield_bond_spread
year,,,,,,,,,,,,
2016-01-01,4.241817,1940.24,2238.83,0.153893,4.348064,0.714958,0.117731,0.003161,0.690550,0.103651,0.053097,3.551266
2017-01-01,4.275067,2278.87,2673.61,0.173217,4.364782,0.784041,0.216055,0.009341,2.801716,0.097239,0.062131,1.473350
2018-01-01,4.577463,2823.81,2506.85,-0.112246,4.491472,1.346223,-0.042269,0.019363,-0.016692,-0.027626,0.045327,4.594155
2019-01-01,4.621887,2704.10,3230.78,0.194771,4.149165,1.118838,0.312117,0.020625,9.635631,0.153295,0.036916,-5.013744
2020-01-01,3.248145,3225.52,3756.07,0.164485,3.901934,0.831404,0.180232,0.003547,11.331898,0.104115,0.103461,-8.083753


 ### Yield spread
 The difference between earnings yield and 10Y US T-Bond is often referred to as the  the "yield spread" and can indicate whether stocks are relatively cheap or expensive compared to bonds. A positive spread suggests stocks may be undervalued (higher earnings yield than bond yield), while a negative spread suggests stocks may be overvalued.

In [15]:
# Filter for the last 40 years
df_last_40 = df_merged[df_merged.index >= '1985-01-01']

# Create line chart for yield-bond spread
fig = px.line(df_last_40, x=df_last_40.index, y='yield_bond_spread', 
              title='S&P 500 Earnings Yield - 10Y US T-Bond Spread (Last 40 Years)',
              labels={'yield_bond_spread': 'Spread (%)', 'x': 'Year'})

# Add a horizontal line at 0 for reference
fig.add_hline(y=0, line_dash="dash", line_color="red", annotation_text="Zero Spread")

fig.show()

In [21]:
# Create a new dataframe with selected columns
df_selected = df_merged[['yield_bond_spread', 'avg_earningyield', '10YUSTBond']]

df_selected.tail(20)

,yield_bond_spread,avg_earningyield,10YUSTBond
year,,,
2001-01-01,-2.739883,2.832298,5.572181
2002-01-01,-12.306675,2.809725,15.116400
2003-01-01,3.528550,3.903869,0.375319
2004-01-01,0.460755,4.951438,4.490684
2005-01-01,2.515732,5.383265,2.867533
2006-01-01,3.865712,5.826713,1.961001
2007-01-01,-4.910108,5.299814,10.209922
2008-01-01,-16.695897,3.405383,20.101280
2009-01-01,13.028770,1.912075,-11.116695


In [20]:
# Plot avg_earningyield, 10YUSTBond, and yield_bond_spread together (since 2000)
start = pd.to_datetime('2000', format='%Y')
df_plot = df_selected[df_selected.index >= start].reset_index()

fig = px.line(
    df_plot,
    x='year',
    y=['avg_earningyield', '10YUSTBond', 'yield_bond_spread'],
    title='Avg Earnings Yield, 10‑Year U.S. Treasury Bond Yield and Spread (since 2000)',
    labels={
        'year': 'Year',
        'value': 'Percentage (%)',
        'variable': 'Metric'
    }
)
# add markers for readability
fig.update_traces(mode='lines+markers')
# adjust layout for clarity
fig.update_layout(
    legend_title_text='Metric',
    yaxis=dict(tickformat=',.2f'),
    xaxis=dict(showgrid=True),
    template='plotly_white',
    margin=dict(l=60, r=20, t=60, b=40)
)
fig.show()